# Video Game Sales: Data Cleaning & Exploratory Analysis
### Industry Trends (1980 - 2024)
**Name:** Shubham Singh


**Objective:** To clean a raw dataset of 16,000+ video games and prepare it for visualization in Power BI.

## 1. Enivironment Setup
In this section, we import the required libraries data cleaning and analysis.

In [1]:
import pandas as pd
import numpy as np

## 2. Loading the CSV file
Loading the data and setting the index name to 'G_id'(Game Id) to serve as a Primary Key.

In [2]:
vg_filepath = '../data/vgsales_raw.csv'
df = pd.read_csv(vg_filepath)
df.index.name = 'G_id'

In [3]:
df.head()

,img,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
G_id,,,,,,,,,,,,,,
0,/games/boxart/full_6510540AmericaFrontccc.jpg,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,17-09-2013,NaN
1,/games/boxart/full_5563178AmericaFrontccc.jpg,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,18-11-2014,03-01-2018
2,/games/boxart/827563ccc.jpg,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,28-10-2002,NaN
3,/games/boxart/full_9218923AmericaFrontccc.jpg,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,NaN,15.86,9.06,0.06,5.33,1.42,17-09-2013,NaN
4,/games/boxart/full_4990510AmericaFrontccc.jpg,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,06-11-2015,14-01-2018


## 3. Initial Data Investigation
Before cleaning, We have to identify the data type inconsistencies, structural errors and missing or null values in rows.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64016 entries, 0 to 64015
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   img           64016 non-null  str    
 1   title         64016 non-null  str    
 2   console       64016 non-null  str    
 3   genre         64016 non-null  str    
 4   publisher     64016 non-null  str    
 5   developer     63999 non-null  str    
 6   critic_score  6678 non-null   float64
 7   total_sales   18922 non-null  float64
 8   na_sales      12637 non-null  float64
 9   jp_sales      6726 non-null   float64
 10  pal_sales     12824 non-null  float64
 11  other_sales   15128 non-null  float64
 12  release_date  56965 non-null  str    
 13  last_update   17879 non-null  str    
dtypes: float64(6), str(8)
memory usage: 6.8 MB


In [5]:
sales = ['na_sales', 'jp_sales', 'pal_sales', 'other_sales']
df['calc_total'] = df[sales].sum(axis = 1)
print(df[['calc_total', 'total_sales']].head())

      calc_total  total_sales
G_id                         
0          20.33        20.32
1          19.39        19.39
2          16.15        16.15
3          15.87        15.86
4          15.08        15.09


### **3.1 Data Integrity Check: Sales Verification**
To ensure the reliability of the dataset, I performed a row-wise summation of the regional sales (`na_sales`, `jp_sales`, `pal_sales`, `other_sales`) and compared the result against `total_sales`.

**Observations:**
* **Consistency:** The calculated totals match the provided totals for the vast majority of the records.
* **Minor Discrepancies:** A variance of approximately 0.01 was observed in several rows (e.g. G_id 0 with +0.01 error). 
* **Conclusion:** These inconsistencies are likely due to floating-point precision errors or rounding during the initial data entry phase. The variance is negligible ($<0.1\%$) and does not impact the overall integrity of the analysis.

In [6]:
df.isnull().sum()

img                 0
title               0
console             0
genre               0
publisher           0
developer          17
critic_score    57338
total_sales     45094
na_sales        51379
jp_sales        57290
pal_sales       51192
other_sales     48888
release_date     7051
last_update     46137
calc_total          0
dtype: int64

### **3.2 Null Value Assessment**
The `isnull().sum()` function several gaps that directly impact the analysis. Following is the breakdown of the missing data:

**Critical Impact (Sales & Timeline):**
* **Total_sales / Regional Sales:** Approximate 45k and 50k missing values respectively. Missing sales data for these records makes them unusable for market share analysis.
* **Release_date / Last_update:** 7051 and 46137 missing values respectively. This will create gaps in time-series visualizations.

**Metadata & Contextual Impact:**
* **Developer:** 17 missing values. This limits our ability to analyze studio-specific performance.
* **Critic_score:** 57338 missing values. **Note:** It is common for older or niche titles to lack professional review scores; we will need a strategy to handle these without skewing the average.

**Conclusion:**
If left as it is, these nulls or missing values will result in "Unknown" categories or broken calculations in Power BI, significantly lowering the professional quality of the final dashboard.

## **3.2 Clean-up: Removing Redundant Columns**
Now that the data integrity check is complete and the regional sales have been verified against the total sales, the temporary calculation column (`calc_total`) is no longer needed. 
Removing `img` column as it doesn't offer anything to the EDA.

**Action:** Dropping the redundant column to keep the dataset clean and ready for the next cleaning phase.

In [7]:
df.drop(columns = ['calc_total'], inplace = True)

# Dropping the image column to reduce file size and remove broken links
df.drop(columns=['img'], inplace=True, errors='ignore')

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64016 entries, 0 to 64015
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         64016 non-null  str    
 1   console       64016 non-null  str    
 2   genre         64016 non-null  str    
 3   publisher     64016 non-null  str    
 4   developer     63999 non-null  str    
 5   critic_score  6678 non-null   float64
 6   total_sales   18922 non-null  float64
 7   na_sales      12637 non-null  float64
 8   jp_sales      6726 non-null   float64
 9   pal_sales     12824 non-null  float64
 10  other_sales   15128 non-null  float64
 11  release_date  56965 non-null  str    
 12  last_update   17879 non-null  str    
dtypes: float64(6), str(7)
memory usage: 6.3 MB


In [9]:
df.describe()

,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales
count,6678.000000,18922.000000,12637.000000,6726.000000,12824.000000,15128.000000
mean,7.220440,0.349113,0.264740,0.102281,0.149472,0.043041
std,1.457066,0.807462,0.494787,0.168811,0.392653,0.126643
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,6.400000,0.030000,0.050000,0.020000,0.010000,0.000000
50%,7.500000,0.120000,0.120000,0.040000,0.040000,0.010000
75%,8.300000,0.340000,0.280000,0.120000,0.140000,0.030000
max,10.000000,20.320000,9.760000,2.130000,9.850000,3.120000


### **3.4 Observation: Temporal Data & String Storage**
During the categorical audit, I realised two critical issues regarding the timeline columns (`Release_date` and `Last_update`):

1.  **Dtype Mismatch:** These columns are currently stored as `object` (strings). This prevents the calculation of "oldest/newest" games and blocks time-series plotting.
2.  **High Null Volume:**
       * `Release_date`: 7,051 missing values.
       * `Last_update`: 46,137 missing values.
4.  **Impact:** `Last_update` is too "empty" to be a primary metric. I will prioritize `Release_date` for the Power BI timeline but must convert it to a true `datetime` format first.

In [10]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
critic_score,6678.0,7.22,1.46,1.0,6.40,7.50,8.30,10.00
total_sales,18922.0,0.35,0.81,0.0,0.03,0.12,0.34,20.32
na_sales,12637.0,0.26,0.49,0.0,0.05,0.12,0.28,9.76
jp_sales,6726.0,0.10,0.17,0.0,0.02,0.04,0.12,2.13
pal_sales,12824.0,0.15,0.39,0.0,0.01,0.04,0.14,9.85
other_sales,15128.0,0.04,0.13,0.0,0.00,0.01,0.03,3.12


In [11]:
df.describe(include=['object', 'string'])

,title,console,genre,publisher,developer,release_date,last_update
count,64016,64016,64016,64016,63999,56965,17879
unique,39798,81,20,3383,8862,7922,1545
top,Plants vs. Zombies,PC,Misc,Unknown,Unknown,01-01-1994,06-01-2018
freq,17,12617,9304,8842,4435,515,165


## 4. Data Cleaning & Transformation
This section focuses on "cleaning the slate"—removing unusable records and converting data types into formats that support advanced analysis.

### **4.1 Cleaning Overview**
    
| Column(s) | Detected Issue | Action Taken | Reasoning |
| :--- | :--- | :--- | :--- |
| **Release_date** | String Dtype | `pd.to_datetime` | Enables time-series analysis and chronological sorting in Power BI. |
| **Total_sales** | 45k+ Null Values | `dropna()` | Sales is our primary metric; records without sales data provide no analytical value. |
| **Developer** | 17 Null Values | `fillna("Unknown")` | Preserves valid sales data while labeling missing information as `Unknown`. |
| **Critic_score** | 57k+ Null Values | Keep as `NaN` | Filling with 0 would warp the final analysis; these will be filtered during the visualization phase. |


In [12]:
df.release_date = pd.to_datetime(df['release_date'], dayfirst = True, errors = 'coerce')
df.last_update = pd.to_datetime(df['release_date'], dayfirst = True, errors = 'coerce')

print("Date Conversion results")
print(df[['release_date' , 'last_update']].dtypes)
print(f"Dataset Timeline: {df['release_date'].min().year} to {df['last_update'].max().year}")

Date Conversion results
release_date    datetime64[us]
last_update     datetime64[us]
dtype: object
Dataset Timeline: 1971 to 2024


In [13]:
# Only keep data where the release year is 2020 or earlier
df = df[df['release_date'].dt.year <= 2020]

df['last_update'] = pd.to_datetime(df['last_update'])

### **4.2 Filtering Missing Sales Data**
With the dataset spanning over 50 years of gaming history, maintaining data integrity is paramount. I am removing records that lack `total_sales` data, as these "ghost records" would skew market share analysis and provide no insights for the final dashboard.

In [14]:
before_drop = df.shape[0]

df.dropna(subset = ['total_sales'], inplace = True)

after_drop = df.shape[0]
print(f"Result: Removed {before_drop - after_drop} unusable records.")
print(f"Current Dataset Size: {after_drop} rows.")

Result: Removed 37413 unusable records.
Current Dataset Size: 18832 rows.


### 4.3 Refining the Target Metric (Zero-Value Filtering)
During the cleaning process, I identified approximately 1,300 records where total_sales was exactly 0.00. While these are technically not "null," they represent games with no commercial tracking data.

To ensure the final Power BI dashboard reflects actual market trends and provides accurate average revenue metrics, I have filtered these records out. The final dataset now consists of games with confirmed commercial activity.

In [15]:
df = df[df['total_sales'] > 0]

print(f"High-quality records remaining: {df.shape[0]}")

High-quality records remaining: 17508


In [16]:
df.developer = df['developer'].fillna('Unknown')

print("Final Null Audit:")
print(df[['total_sales', 'release_date', 'developer']].isnull().sum())

Final Null Audit:
total_sales     0
release_date    0
developer       0
dtype: int64


In [17]:
df.dropna(subset=['release_date'], inplace=True)

print(f"Final dataset size: {df.shape[0]} rows.")

# 3. The "Clean Slate" verification
print("\nFinal Null Count for all columns:")
print(df.isnull().sum())

Final dataset size: 17508 rows.

Final Null Count for all columns:
title               0
console             0
genre               0
publisher           0
developer           0
critic_score    13531
total_sales         0
na_sales         5065
jp_sales        11067
pal_sales        5755
other_sales      3404
release_date        0
last_update         0
dtype: int64


### **4.4 Handling Sparse Metadata (Critic Scores)**
During the final audit, I observed that approximately 13,000 records lack a `Critic_score`. 

* **Strategy:** I have chosen **not** to drop these records or fill them with a placeholder (like 0 or the mean). 
* **Reasoning:** Dropping these would result in the loss of ~74% of valid commercial sales data. 
* **Execution:** These will remain as `NaN` (Not a Number). Power BI's aggregation engines will naturally exclude these nulls when calculating average scores, ensuring that the "Sales vs. Rating" analysis remains statistically sound without sacrificing the volume of the primary dataset.

In [18]:
# Final confirmation of the 'Big Three'
clean_check = df[['total_sales', 'release_date', 'developer']].isnull().sum()

print("--- Final Data Integrity Report ---")
print(clean_check)
print(f"\nTotal High-Quality Rows: {df.shape[0]}")
print(f"Total Rows with Critic Scores: {df['critic_score'].notnull().sum()}")

--- Final Data Integrity Report ---
total_sales     0
release_date    0
developer       0
dtype: int64

Total High-Quality Rows: 17508
Total Rows with Critic Scores: 3977


In [19]:
df.info()

<class 'pandas.DataFrame'>
Index: 17508 entries, 0 to 17569
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   title         17508 non-null  str           
 1   console       17508 non-null  str           
 2   genre         17508 non-null  str           
 3   publisher     17508 non-null  str           
 4   developer     17508 non-null  str           
 5   critic_score  3977 non-null   float64       
 6   total_sales   17508 non-null  float64       
 7   na_sales      12443 non-null  float64       
 8   jp_sales      6441 non-null   float64       
 9   pal_sales     11753 non-null  float64       
 10  other_sales   14104 non-null  float64       
 11  release_date  17508 non-null  datetime64[us]
 12  last_update   17508 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(6), str(5)
memory usage: 1.9 MB


### **4.5 Conclusion: Data Readiness Statement**
The data cleaning and transformation phase is now complete. By implementing a rigorous filtering strategy and correcting structural inconsistencies, the dataset has been refined from its raw state into a high-quality analytical asset.

**Final Data Profile:**
* **Time Span:** 1971 – 2024 (Full industry lifecycle). Almost 50 years!
* **Integrity:** 100% of analyzed records contain valid `total_sales`, `Release_date`, and `Developer` data.
* **Dimensionality:** Reduced from ~64k records to approximately **17.5k** high-impact commercial records.
* **Status:** The dataset is now normalized and exported as `vgsales_final_cleaned.csv`, ready for Exploratory Data Analysis (EDA) and Power BI visualization.

In [20]:
# The final snapshot of your project's foundation
print("--- FINAL DATASET SUMMARY ---")
df.info()

# A quick look at the first 5 rows of your high-quality data
df.head()

--- FINAL DATASET SUMMARY ---
<class 'pandas.DataFrame'>
Index: 17508 entries, 0 to 17569
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   title         17508 non-null  str           
 1   console       17508 non-null  str           
 2   genre         17508 non-null  str           
 3   publisher     17508 non-null  str           
 4   developer     17508 non-null  str           
 5   critic_score  3977 non-null   float64       
 6   total_sales   17508 non-null  float64       
 7   na_sales      12443 non-null  float64       
 8   jp_sales      6441 non-null   float64       
 9   pal_sales     11753 non-null  float64       
 10  other_sales   14104 non-null  float64       
 11  release_date  17508 non-null  datetime64[us]
 12  last_update   17508 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(6), str(5)
memory usage: 1.9 MB


,title,console,genre,publisher,developer,critic_score,total_sales,na_sales,jp_sales,pal_sales,other_sales,release_date,last_update
G_id,,,,,,,,,,,,,
0,Grand Theft Auto V,PS3,Action,Rockstar Games,Rockstar North,9.4,20.32,6.37,0.99,9.85,3.12,2013-09-17,2013-09-17
1,Grand Theft Auto V,PS4,Action,Rockstar Games,Rockstar North,9.7,19.39,6.06,0.60,9.71,3.02,2014-11-18,2014-11-18
2,Grand Theft Auto: Vice City,PS2,Action,Rockstar Games,Rockstar North,9.6,16.15,8.41,0.47,5.49,1.78,2002-10-28,2002-10-28
3,Grand Theft Auto V,X360,Action,Rockstar Games,Rockstar North,NaN,15.86,9.06,0.06,5.33,1.42,2013-09-17,2013-09-17
4,Call of Duty: Black Ops 3,PS4,Shooter,Activision,Treyarch,8.1,15.09,6.18,0.41,6.05,2.44,2015-11-06,2015-11-06


### 4.6 Final Data Integrity Check
Before proceeding to the visual analysis, a final check was performed to ensure that no null values remained in the critical features (Sales and Critic Scores). 
In this final pre-processing step, I performed a complete sweep of the dataset to ensure it is ready for visualization in Power BI. By doing a full investigation, I found out that the sales(all sales columns) and critic_score columns still have some values missing or just have somehow null values.
**Handling Strategy:**
1. **Regional Sales:** All missing values were filled with '0', it was done so, to make it feel like a particular game was just not released in that specific region at that time. 
2. **Critic Scores:** I utilized **Mean Imputation** to fill missing scores. This allows us to retain the sales data for these records without introducing bias or skewing the statistical correlation in Section 5.5.

**NOTE: Mean Imputation utilizes aggregated or average value of the entire feature or attribute to replace the missing value**

In [21]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
title               0
console             0
genre               0
publisher           0
developer           0
critic_score    13531
total_sales         0
na_sales         5065
jp_sales        11067
pal_sales        5755
other_sales      3404
release_date        0
last_update         0
dtype: int64


In [22]:
sales_cols = ['na_sales', 'pal_sales', 'jp_sales', 'other_sales', 'total_sales']
df[sales_cols] = df[sales_cols].fillna(0)

avg_score = df['critic_score'].mean()
df['critic_score'] = df['critic_score'].fillna(avg_score)

print("Missing values after final sweep:")
print(df.isnull().sum())

df.head()

# Exporting the clean file
df.to_csv('vgsales_final_cleaned.csv', index=False)

print("File Saved: vgsales_final_cleaned.csv is ready for analysis!")

Missing values after final sweep:
title           0
console         0
genre           0
publisher       0
developer       0
critic_score    0
total_sales     0
na_sales        0
jp_sales        0
pal_sales       0
other_sales     0
release_date    0
last_update     0
dtype: int64
File Saved: vgsales_final_cleaned.csv is ready for analysis!


## 5. Exploratory Data Analysis (EDA)
In this section, I explore the cleaned dataset to uncover market trends, console's dominance, and the evolution of gaming over the last five decades. This analysis serves as the foundation for the final Power BI dashboard.

### 5.1 The "Heavy Hitters": Top 10 Games by Global Sales
Before diving into complex trends, let's identify the absolute champions of the industry.

In [23]:
top_10_games = df.nlargest(10, 'total_sales')[['title', 'console', 'total_sales', 'release_date']]

print("--- Top 10 Best Selling Games of All Time ---")
print(top_10_games)

--- Top 10 Best Selling Games of All Time ---
                               title console  total_sales release_date
G_id                                                                  
0                 Grand Theft Auto V     PS3        20.32   2013-09-17
1                 Grand Theft Auto V     PS4        19.39   2014-11-18
2        Grand Theft Auto: Vice City     PS2        16.15   2002-10-28
3                 Grand Theft Auto V    X360        15.86   2013-09-17
4          Call of Duty: Black Ops 3     PS4        15.09   2015-11-06
5     Call of Duty: Modern Warfare 3    X360        14.82   2011-11-08
6            Call of Duty: Black Ops    X360        14.74   2010-11-09
7              Red Dead Redemption 2     PS4        13.94   2018-10-26
8         Call of Duty: Black Ops II    X360        13.86   2012-11-13
9         Call of Duty: Black Ops II     PS3        13.80   2012-11-13


In [24]:
top_10_games.index = range(1,11)
top_10_games

,title,console,total_sales,release_date
1,Grand Theft Auto V,PS3,20.32,2013-09-17
2,Grand Theft Auto V,PS4,19.39,2014-11-18
3,Grand Theft Auto: Vice City,PS2,16.15,2002-10-28
4,Grand Theft Auto V,X360,15.86,2013-09-17
5,Call of Duty: Black Ops 3,PS4,15.09,2015-11-06
6,Call of Duty: Modern Warfare 3,X360,14.82,2011-11-08
7,Call of Duty: Black Ops,X360,14.74,2010-11-09
8,Red Dead Redemption 2,PS4,13.94,2018-10-26
9,Call of Duty: Black Ops II,X360,13.86,2012-11-13
10,Call of Duty: Black Ops II,PS3,13.80,2012-11-13


### 5.2 The "Golden Age" of Gaming (Sales Over Time)
Since I spent alot of time cleaning the sales and dates of games. I will be using it to find out the most profitable year or era for the gaming industry.

In [25]:
yearly_sales = df.groupby(df['release_date'].dt.year)['total_sales'].sum()

peak_year = yearly_sales.idxmax()
peak_value = yearly_sales.max()

print(f"The most profitable year in gaming history was {int(peak_year)} with ${peak_value:.2f}M in sales.")

The most profitable year in gaming history was 2008 with $538.11M in sales.


## 5.3 What Actually Sells? (Genre Popularity)
In this section, I transition from individual titles to broader market segments. By analyzing total revenue by genre, we can identify which categories of games have the highest commercial "ceiling" and most consistent market demand.

**Analysis Goals:**
* Identify the top-performing genres by global revenue.
* Understand market distribution across different gameplay styles.

> **[INSERT BAR CHART: TOTAL SALES BY GENRE]**

**Initial Observations:**
The distribution reveals that certain genres, such as **Action** and **Sports**, command a massive share of the market. This often correlates with high-replayability factors and consistent annual releases from major franchises.

In [26]:
top_genre = df.groupby('genre').total_sales.sum().sort_values(ascending = False).head(5)
print(top_genre)

genre
Sports     1186.77
Action     1124.95
Shooter     995.47
Misc        557.55
Racing      523.51
Name: total_sales, dtype: float64


## 5.4 Critic Scores vs. Sales (Does quality = money?)
I calculated the **Pearson Correlation Coefficient** between `critic_score` and `total_sales` to test whether professional critical reception serves as a reliable predictor of commercial success.

**Result:** I obtained a correlation value of **0.29**.

> **[INSERT SCATTER/REGRESSION PLOT: CRITIC SCORE VS. TOTAL SALES]**

**Analysis:**
This result indicates a **weak positive correlation**. Statistically, this suggests that while higher critical acclaim can contribute to sales, it is not the primary driver of revenue in the gaming industry.

> [!IMPORTANT]
> **The Fanbase Anomaly:** High-loyalty franchises (e.g., *Yakuza* or *GTA*) often see massive sales regardless of critical reception. This suggests that brand equity is a stronger driver of revenue than critic consensus.

In [27]:
# Simple correlation check
correlation = df['total_sales'].corr(df['critic_score'])
print(f"Correlation between Score and Sales: {correlation:.2f}")

Correlation between Score and Sales: 0.24


## **5.5 Market Giants: Top Publishers Analysis**

In this final analysis, I shifted the focus from individual titles to the major players driving the industry's revenue. Instead of analyzing hundreds of small studios, I grouped the data into key **"Market Giants"** to identify which publishers dominate the global software market.

**Technical Approach:**
* **Custom Column:** I created a custom `Company` column using `np.select` to categorize publishers. This allowed me to isolate industry leaders like Nintendo, Sony, Microsoft, Rockstar Games and grouped together small studios as Other Third Party.
* **Aggregation:** I used a **Pivot Table** to calculate the total global revenue for these giants on an annual basis, grouping all other studios into a "Third-Party" category for clarity.

> **[INSERT AREA CHART: SALES BY COMPANY OVER TIME]**

**Key Insight:**
The visualization highlights the massive market share held by a few specific publishers. While platform holders like **Nintendo** show high software sales, the inclusion of **Rockstar Games** as a standalone company demonstrates the immense impact a single "AAA" studio can have on annual industry revenue.


In [28]:
conditions = [
    df['publisher'] == 'Nintendo',
    df['publisher'].isin(['Sony Computer Entertainment', 'Sony Interactive Entertainment']),
    df['publisher'].isin(['Microsoft Game Studios', 'Xbox Game Studios']),
    df['publisher'] == 'Rockstar Games' # <--- Added Rockstar specifically
]

choices = ['Nintendo', 'Sony', 'Microsoft', 'Rockstar']

df['Company'] = np.select(conditions, choices, default='Other Third-Party')

In [29]:
console_wars = df.pivot_table(index=df['release_date'].dt.year, 
                                  columns='Company', 
                                  values='total_sales', 
                                  aggfunc='sum')

# Let's see the 'Golden Era' of the war (2000 - 2015)
print(console_wars.loc[2000:2015])

Company       Microsoft  Nintendo  Other Third-Party  Rockstar   Sony
release_date                                                         
2000                NaN      3.75             143.15      4.44  19.78
2001                NaN      7.38             184.10     18.16  17.11
2002               3.01      2.83             275.24     18.00  15.42
2003               6.21      4.89             267.70     10.02  12.07
2004               0.77     15.80             248.44      2.43  17.24
2005               4.00     13.98             253.98     19.97  21.55
2006               0.80      7.99             209.56     10.73  15.34
2007              11.38      7.14             390.70      3.92  23.25
2008               4.59      3.12             480.76     30.17  19.47
2009               6.27      5.92             451.49      5.15  26.53
2010              20.93      8.31             396.92     10.97  16.89
2011               0.68      4.32             407.77     12.57  14.98
2012                